# PuijaiRiskNLP — Fine-tune WangchanBERTa & Run Experiments
รองรับทั้งการรันบนเครื่อง (**VS Code / Jupyter Local**) และบน **Google Colab (GPU T4)**

- **แต่ละเซลล์ทำงานได้อิสระ (Self-contained):** สามารถกด **Run All** หรือกดรันทีละเซลล์จากบนลงล่างได้เลย
- หากมีไฟล์ข้อมูลจริงที่ติดป้ายแล้วที่ `data/processed/conversations.csv` โน้ตบุ๊กจะใช้ไฟล์นั้นโดยอัตโนมัติ (หากยังไม่มีจะใช้ `data/sample/sample_conversations.csv`)

In [15]:
# [Cell 1] ตั้งค่า Path และติดตั้งแพ็กเกจที่จำเป็น
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
for extra_pkg in ("D:/miniconda3/Lib/site-packages", "D:/puijai_env/site-packages"):
    if Path(extra_pkg).exists() and str(Path(extra_pkg)) not in sys.path:
        sys.path.insert(0, str(Path(extra_pkg)))

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Working directory: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")

REQUIRED_MODULES = ("torch", "transformers", "sklearn", "matplotlib", "sentencepiece", "puijai_risk")
missing = [m for m in REQUIRED_MODULES if importlib.util.find_spec(m) is None]
if missing:
    print(f"กำลังติดตั้งแพ็กเกจที่ยังขาด: {missing} ...")
    target_args = ["--target", "D:/puijai_env/site-packages"] if Path("D:/puijai_env/site-packages").exists() else []
    subprocess.check_call([sys.executable, "-m", "pip", "install", *target_args, *missing])
    importlib.invalidate_caches()
print("แพ็กเกจพร้อมใช้งานครบถ้วน!")

Working directory: c:\Users\USER\Downloads\แชทบอท
Python executable: c:\Users\USER\Downloads\แชทบอท\.venv\Scripts\python.exe
แพ็กเกจพร้อมใช้งานครบถ้วน!


In [16]:
# [Cell 2] ทดสอบชุด Unit Tests และรัน Keyword Baseline ด้วยข้อมูลตัวอย่าง
import os
import subprocess
import sys
from pathlib import Path

if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
for extra_pkg in ("D:/miniconda3/Lib/site-packages", "D:/puijai_env/site-packages"):
    if Path(extra_pkg).exists() and str(Path(extra_pkg)) not in sys.path:
        sys.path.insert(0, str(Path(extra_pkg)))

def run_cmd(*args: str) -> None:
    res = subprocess.run([sys.executable, *args], text=True, capture_output=True)
    if res.stdout:
        print(res.stdout, end="")
    if res.returncode != 0:
        print(res.stderr, file=sys.stderr)
        raise RuntimeError(f"Command failed (exit {res.returncode}): {' '.join(args)}")

run_cmd("-m", "pytest", "-q")
run_cmd(
    "-m",
    "puijai_risk.evaluation.length_experiment",
    "--classifier",
    "keyword",
    "--data",
    "data/sample/sample_conversations.csv",
)

.............................                                            [100%]
29 passed in 2.30s
[Exp.1] message length buckets -> results\length_experiment_keyword_exp1.csv
   {'bucket': '<5', 'n': 10, 'accuracy': 1.0, 'macro_f1': 1.0, 'critical_precision': 1.0, 'critical_recall': 1.0, 'critical_f1': 1.0, 'warning_recall': 1.0}
   {'bucket': '5-20', 'n': 20, 'accuracy': 0.8, 'macro_f1': 0.8, 'critical_precision': 1.0, 'critical_recall': 0.6667, 'critical_f1': 0.8, 'warning_recall': 0.75}
   {'bucket': '>20', 'n': 6, 'accuracy': 1.0, 'macro_f1': 1.0, 'critical_precision': 1.0, 'critical_recall': 1.0, 'critical_f1': 1.0, 'warning_recall': 1.0}
[Exp.2] context window N -> results\length_experiment_keyword_exp2.csv
   {'n_turns': 1, 'n': 36, 'avg_tokens': 9.56, 'accuracy': 0.8889, 'macro_f1': 0.8862, 'critical_precision': 1.0, 'critical_recall': 0.8, 'critical_f1': 0.8889, 'warning_recall': 0.8333, 'erde_5': 0.1231, 'erde_50': 0.0833}
   {'n_turns': 3, 'n': 36, 'avg_tokens': 18.64, 'acc

In [17]:
# [Cell 3] Fine-tune WangchanBERTa และเทรน TF-IDF Baseline
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
for extra_pkg in ("D:/miniconda3/Lib/site-packages", "D:/puijai_env/site-packages"):
    if Path(extra_pkg).exists() and str(Path(extra_pkg)) not in sys.path:
        sys.path.insert(0, str(Path(extra_pkg)))

if importlib.util.find_spec("torch") is None or importlib.util.find_spec("transformers") is None:
    target_args = ["--target", "D:/puijai_env/site-packages"] if Path("D:/puijai_env/site-packages").exists() else []
    subprocess.check_call([sys.executable, "-m", "pip", "install", *target_args, "torch", "transformers", "accelerate", "sentencepiece"])
    importlib.invalidate_caches()

import torch

def run_cmd(*args: str) -> None:
    res = subprocess.run([sys.executable, *args], text=True, capture_output=True)
    if res.stdout:
        print(res.stdout, end="")
    if res.returncode != 0:
        print(res.stderr, file=sys.stderr)
        raise RuntimeError(f"Command failed (exit {res.returncode}): {' '.join(args)}")

DATA_PATH = Path("data/processed/conversations.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/sample/sample_conversations.csv")
    print(f"ไม่พบ data/processed/conversations.csv -> ใช้ข้อมูลตัวอย่าง {DATA_PATH} แทน")
else:
    print(f"ใช้ข้อมูลจริงจาก {DATA_PATH}")

EPOCHS = "5" if torch.cuda.is_available() else "1"
N_LIST = [1, 3, 5] if torch.cuda.is_available() else [3]
print(f"Device: {'CUDA GPU' if torch.cuda.is_available() else 'CPU'} | epochs={EPOCHS} | n_turns={N_LIST}")

for n in N_LIST:
    print(f"\n=== Training WangchanBERTa (n_turns={n}) ===")
    run_cmd(
        "-m",
        "puijai_risk.training.train_wangchanberta",
        "--data",
        str(DATA_PATH),
        "--output-dir",
        f"models/wcb-n{n}",
        "--n-turns",
        str(n),
        "--epochs",
        EPOCHS,
    )

print("\n=== Training TF-IDF Baseline ===")
run_cmd(
    "-m",
    "puijai_risk.classifiers.tfidf_svm",
    "--data",
    "models/wcb-n3/splits/train.csv",
    "--out",
    "models/tfidf.joblib",
    "--n-turns",
    "3",
)

ไม่พบ data/processed/conversations.csv -> ใช้ข้อมูลตัวอย่าง data\sample\sample_conversations.csv แทน
Device: CPU | epochs=1 | n_turns=[3]

=== Training WangchanBERTa (n_turns=3) ===


Exception in thread Thread-24 (_readerthread):
Traceback (most recent call last):
  File "D:\miniconda3\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "D:\miniconda3\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\miniconda3\Lib\subprocess.py", line 1614, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "D:\miniconda3\Lib\encodings\cp874.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x88 in position 82: character maps to <undefined>


{'train': 27, 'val': 3, 'test': 6}
class weights: [0.6000000238418579, 1.0, 3.0]
{'eval_loss': '0.8661', 'eval_macro_f1': '0.1667', 'eval_critical_recall': '1', 'eval_accuracy': '0.3333', 'eval_selection_score': '0.5833', 'eval_runtime': '0.1515', 'eval_samples_per_second': '19.8', 'eval_steps_per_second': '6.601', 'epoch': '1'}
{'train_runtime': '10.75', 'train_samples_per_second': '2.512', 'train_steps_per_second': '0.186', 'train_loss': '1.026', 'epoch': '1'}
test: {'test_loss': 0.8982340693473816, 'test_macro_f1': 0.3, 'test_critical_recall': 1.0, 'test_accuracy': 0.3333333333333333, 'test_selection_score': 0.65, 'test_runtime': 0.1822, 'test_samples_per_second': 32.937, 'test_steps_per_second': 5.489, 'epoch': 1.0}

=== Training TF-IDF Baseline ===
saved models/tfidf.joblib (27 samples)


In [18]:
# [Cell 4] ประเมินผลเปรียบเทียบทั้ง 3 โมเดล (Keyword, TF-IDF, WangchanBERTa) บน Test Split
import os
import subprocess
import sys
from pathlib import Path

if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
for extra_pkg in ("D:/miniconda3/Lib/site-packages", "D:/puijai_env/site-packages"):
    if Path(extra_pkg).exists() and str(Path(extra_pkg)) not in sys.path:
        sys.path.insert(0, str(Path(extra_pkg)))

def run_cmd(*args: str) -> None:
    res = subprocess.run([sys.executable, *args], text=True, capture_output=True)
    if res.stdout:
        print(res.stdout, end="")
    if res.returncode != 0:
        print(res.stderr, file=sys.stderr)
        raise RuntimeError(f"Command failed (exit {res.returncode}): {' '.join(args)}")

TEST_CSV = Path("models/wcb-n3/splits/test.csv")
if not TEST_CSV.exists():
    TEST_CSV = Path("data/sample/sample_conversations.csv")
    print(f"ยังไม่ได้รัน Cell 3 เพื่อสร้าง test split -> ใช้ {TEST_CSV} ทดสอบแทน")

candidates = [("keyword", [], True)]
candidates.append(("tfidf", ["--model-path", "models/tfidf.joblib"], Path("models/tfidf.joblib").exists()))
candidates.append(("wangchanberta", ["--model-path", "models/wcb-n3"], (Path("models/wcb-n3") / "config.json").exists()))

for clf_name, extra, ready in candidates:
    if not ready:
        print(f"\n[ข้าม {clf_name}] ยังไม่มีไฟล์โมเดล (กรุณารัน Cell 3 ก่อน)")
        continue
    print(f"\n=== Evaluating {clf_name} ===")
    run_cmd(
        "-m",
        "puijai_risk.evaluation.length_experiment",
        "--classifier",
        clf_name,
        *extra,
        "--data",
        str(TEST_CSV),
        "--n-turns",
        "1",
        "3",
        "5",
    )


=== Evaluating keyword ===
[Exp.1] message length buckets -> results\length_experiment_keyword_exp1.csv
   {'bucket': '<5', 'n': 4, 'accuracy': 1.0, 'macro_f1': 1.0, 'critical_precision': 1.0, 'critical_recall': 1.0, 'critical_f1': 1.0, 'warning_recall': 1.0}
   {'bucket': '5-20', 'n': 2, 'accuracy': 1.0, 'macro_f1': 0.6667, 'critical_precision': 0.0, 'critical_recall': 0.0, 'critical_f1': 0.0, 'warning_recall': 1.0}
   {'bucket': '>20', 'n': 0}
[Exp.2] context window N -> results\length_experiment_keyword_exp2.csv
   {'n_turns': 1, 'n': 6, 'avg_tokens': 4.67, 'accuracy': 1.0, 'macro_f1': 1.0, 'critical_precision': 1.0, 'critical_recall': 1.0, 'critical_f1': 1.0, 'warning_recall': 1.0, 'erde_5': 0.0596, 'erde_50': 0.0}
   {'n_turns': 3, 'n': 6, 'avg_tokens': 9.67, 'accuracy': 1.0, 'macro_f1': 1.0, 'critical_precision': 1.0, 'critical_recall': 1.0, 'critical_f1': 1.0, 'warning_recall': 1.0, 'erde_5': 0.0596, 'erde_50': 0.0}
   {'n_turns': 5, 'n': 6, 'avg_tokens': 9.67, 'accuracy': 1.0,

Exception in thread Thread-32 (_readerthread):
Traceback (most recent call last):
  File "D:\miniconda3\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "D:\miniconda3\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\miniconda3\Lib\subprocess.py", line 1614, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "D:\miniconda3\Lib\encodings\cp874.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x88 in position 82: character maps to <undefined>


[Exp.1] message length buckets -> results\length_experiment_wangchanberta_exp1.csv
   {'bucket': '<5', 'n': 4, 'accuracy': 0.25, 'macro_f1': 0.1333, 'critical_precision': 0.0, 'critical_recall': 0.0, 'critical_f1': 0.0, 'warning_recall': 1.0}
   {'bucket': '5-20', 'n': 2, 'accuracy': 0.5, 'macro_f1': 0.2222, 'critical_precision': 0.0, 'critical_recall': 0.0, 'critical_f1': 0.0, 'warning_recall': 1.0}
   {'bucket': '>20', 'n': 0}
[Exp.2] context window N -> results\length_experiment_wangchanberta_exp2.csv
   {'n_turns': 1, 'n': 6, 'avg_tokens': 4.67, 'accuracy': 0.3333, 'macro_f1': 0.1667, 'critical_precision': 0.0, 'critical_recall': 0.0, 'critical_f1': 0.0, 'warning_recall': 1.0, 'erde_5': 0.5, 'erde_50': 0.5}
   {'n_turns': 3, 'n': 6, 'avg_tokens': 9.67, 'accuracy': 0.3333, 'macro_f1': 0.3, 'critical_precision': 0.3333, 'critical_recall': 1.0, 'critical_f1': 0.5, 'warning_recall': 0.5, 'erde_5': 0.2737, 'erde_50': 0.25}
   {'n_turns': 5, 'n': 6, 'avg_tokens': 9.67, 'accuracy': 0.3333

In [19]:
# [Cell 5] บีบอัดโมเดลเป็นไฟล์ wcb-n3.zip เพื่อนำไปใช้กับแอป
import os
import sys
import zipfile
from pathlib import Path

if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
for extra_pkg in ("D:/miniconda3/Lib/site-packages", "D:/puijai_env/site-packages"):
    if Path(extra_pkg).exists() and str(Path(extra_pkg)) not in sys.path:
        sys.path.insert(0, str(Path(extra_pkg)))

model_dir = Path("models/wcb-n3")
zip_path = Path("D:/puijai_env/wcb-n3.zip") if Path("D:/puijai_env").exists() else Path("wcb-n3.zip")

if not (model_dir / "config.json").exists():
    print("ยังไม่มีโมเดลใน models/wcb-n3 กรุณารัน [Cell 3] เพื่อเทรนโมเดลก่อนครับ")
else:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path in model_dir.rglob("*"):
            if "checkpoints" in file_path.parts or not file_path.is_file():
                continue
            zf.write(file_path, file_path.relative_to(model_dir.parent))

    print(f"สร้างไฟล์โมเดลสำเร็จ: {zip_path.resolve()} ({zip_path.stat().st_size / (1024*1024):.1f} MB)")
    try:
        from google.colab import files  # type: ignore
        files.download(str(zip_path))
    except ImportError:
        print("รันอยู่บนเครื่อง Local — โมเดลพร้อมใช้งานที่โฟลเดอร์:", model_dir.resolve())

สร้างไฟล์โมเดลสำเร็จ: D:\puijai_env\wcb-n3.zip (373.0 MB)
รันอยู่บนเครื่อง Local — โมเดลพร้อมใช้งานที่โฟลเดอร์: D:\puijai_env\models\wcb-n3
